In [ ]:
| **Retriever Type**                   | **Purpose**                                                                         | **Ideal Usage Scenario**                                                                                                       |
| ------------------------------------ | ----------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------ |
| **VectorStoreRetriever**             | Retrieve documents based on vector similarity search (e.g. FAISS, Chroma, Pinecone) | Basic RAG (Retrieval-Augmented Generation) setup for fetching semantically relevant documents                                  |
| **ContextualCompressionRetriever**   | Compresses retrieved docs using an LLM to return only the most relevant parts       | When using large context docs (PDFs, transcripts) where only a portion of each doc is relevant                                 |
| **MultiQueryRetriever**              | Generates multiple rephrased queries to improve retrieval coverage                  | When a user query can be ambiguous or phrased differently; improves diversity of retrieved documents                           |
| **ParentDocumentRetriever**          | Retrieves a full parent document instead of the split chunk                         | When a chunk is matched but full document context is required (e.g., blog, article, contract analysis)                         |
| **BM25Retriever**                    | Keyword-based retriever using classic BM25 algorithm                                | When you want to retrieve based on exact keyword matches (useful for legal, medical, or short documents with precise language) |
| **EnsembleRetriever**                | Combines multiple retrievers (e.g., vector + BM25) with weighted scoring            | When you want to merge semantic and keyword-based search for better hybrid performance                                         |
| **TimeWeightedVectorStoreRetriever** | VectorStoreRetriever + time-based decay to prefer recent documents                  | Best for chatbot memory or news search where **recency** of information is important                                           |
| **TavilySearchAPIRetriever**         | Uses external web search engine via Tavily API to retrieve **live web data**        | When querying real-world events, breaking news, or updated information outside your local docs or vector database              |


In [8]:
#!pip install google.api_core

In [6]:
from google.api_core.exceptions import ResourceExhausted
import time

def ask_with_retry(question, retries=3, delay=20):
    for attempt in range(retries):
        try:
            return ask(question)
        except Exception as e:
            if "RESOURCE_EXHAUSTED" in str(e) and attempt < retries - 1:
                print(f"Rate limited, waiting {delay}s...")
                time.sleep(delay)
            else:
                raise

In [7]:
# RAG (VectorStoreRetriever) tool wrapped in a LangChain v1 agent, using Gemini
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver

# 1. Setup
load_dotenv(".env")
google_api_key = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0, google_api_key=google_api_key)

# 2. Vector DB
with open("sample.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
texts = splitter.split_text(text_data)

embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=google_api_key)
vectorstore = FAISS.from_texts(texts, embedding)
retriever = vectorstore.as_retriever()

# 3. RAG chain (LCEL) — ConversationalRetrievalChain is deprecated;
# this is the current pattern: retrieve -> stuff into prompt -> ask the LLM
rag_prompt = ChatPromptTemplate.from_template(
    "Answer the question using only the context below.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)

def format_docs(docs) -> str:
    return "\n\n".join(d.page_content for d in docs)

def rag_answer(question: str) -> str:
    docs = retriever.invoke(question)
    context = format_docs(docs)
    chain = rag_prompt | llm
    return chain.invoke({"context": context, "question": question}).content

# 4. Wrap RAG as a tool
@tool
def rag_qa(question: str) -> str:
    """Use this to answer questions about LangChain using the loaded documents."""
    return rag_answer(question)

# 5. Memory (checkpointer, not ConversationBufferMemory)
memory = MemorySaver()

# 6. Create agent
agent_executor = create_agent(
    model=llm,
    tools=[rag_qa],
    checkpointer=memory,
)

config = {"configurable": {"thread_id": "rag-conversation-1"}}

def ask(question: str):
    result = agent_executor.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=config,
    )
    return result["messages"][-1].content

# 7. Run conversation
print("1️⃣ First question")
res1 = ask("What is LangChain?")
print("Answer:", res1)

print("\n2️⃣ Follow-up")
res2 = ask("Who created it?")
print("Answer:", res2)

print("\n3️⃣ Ask again")
res3 = ask("Explain LangChain again simply.")
print("Answer:", res3)

1️⃣ First question
Answer: LangChain is a framework designed for developing applications that leverage large language models (LLMs). It offers tools for linking LLM calls, incorporating retrieval-augmented generation (RAG), connecting to vector stores, and constructing agents capable of utilizing external tools. It's a popular framework for creating various LLM-powered applications, such as chatbots, question-answering systems, and autonomous agents.

2️⃣ Follow-up
Answer: LangChain was created by Harrison Chase.

3️⃣ Ask again
Answer: [{'type': 'text', 'text': 'LangChain is a toolkit that helps you build applications using powerful AI language models. It makes it easier to connect these models, add external information, and create smart programs that can use different tools. Think of it as a set of building blocks for making AI apps like chatbots or smart assistants.', 'extras': {'signature': 'Cs8CARFNMg/r/ywKcmxVn6Y9SR1F1ze5I7LYiKfkCoJzD2HBaWF3ZnK0uIEckuEY67ysouCyWCyXho64quVFM+fxgrqM